# 1. Data Extraction

In [1]:
# Imports
import os
import pandas as pd
import numpy as np

import json

import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['figure.figsize'] = (10, 5)
mpl.rcParams['axes.grid'] = False

#%matplotlib ipympl

In [2]:
global ModBus1_info,ModBus15_info,ModBus1_units,ModBus15_units,ModBus1_address
global ModBus15_address,ModBus1_UnitDivision,ModBus15_UnitDivision,ADC_info

ModBus1_info = ['Z-Axis RMS Velocity(in/sec)','Z-Axis RMS Velocity(mm/sec)','Temperature(DegF)','Temperature(DegC)','X-Axis RMS Velocity(in/sec)',
                'X-Axis RMS Velocity(mm/sec)','Z-Axis Peak Acceleration(G)','X-Axis Peak Acceleration(G)','Z-Axis Peak Velocity Component Frequency(Hz)','X-Axis Peak Velocity Component Frequency(Hz)',
                'Z-Axis RMS Acceleration(G)','X-Axis RMS Acceleration(G)','Z-Axis Kurtosis','X-Axis Kurtosis','Z-Axis Crest Factor',
                'X-Axis Crest Factor','Z-Axis Peak Velocity(in/sec)', 'Z-Axis Peak Velocity(mm/sec)','X-Axis Peak Velocity(in/sec)', 'X-Axis Peak Velocity(mm/sec)',
                'Z-Axis High-Frequency RMS Acceleration(G)','Z-Axis High-Frequency RMS Acceleration(G)']

ModBus15_info = ['RMS Voltage L1(Volts)','RMS Voltage L2(Volts)','RMS Voltage L3(Volts)','RMS Current L1(Amps)','RMS Current L2(Amps)',
                'RMS Current L3(Amps)','Phase Shift L1(Degrees)','Phase Shift L2(Degrees)','Phase Shift L3(Degrees)','Frequency L1 Current(Hz)',
                'Frequency L2 Current','Frequency L3 Current(Hz)','Frequency L1 Voltage(Hz)','Frequency L1 Voltage(Hz)','Frequency L1 Voltage(Hz)',
                '15V sense','VIN sense']

ModBus1_units = ['in/sec','mm/sec','DegF','DegC','in/sec',
                'mm/sec','G','G','Hz','Hz',
                'G','G','Kurtosis','Kurtosis','Crest Factor',
                'Crest Factor','in/sec','mm/sec','in/sec','mm/sec',
                'G','G']

ModBus15_units = ['volts','volts','volts','Amps','Amps',
                'Amps','Degrees','Degrees','Degrees','Hz',
                'Hz','Hz','Hz','Hz','Hz',
                'millivolts','millivolts']

ADC_info = ['GND','Temp1(DegC)','Temp2(DegC)','Temp3(DegC)','NAN',
                'NAN','NAN','NAN','NAN','NAN',
                'NAN','NAN','NAN','NAN','NAN']

ModBus1_address = ["405201", "405202", "405203", "405204", "405205", 
                   "405206", "405207", "405208", "405209", "405210", 
                   "405211", "405212", "405213", "405214", "405215", 
                   "405216", "405217", "405218", "405219", "405220",
                     "405221", "405222"]

ModBus15_address = ["300001", "300002", "300003", "300004", "300005", 
                   "300006", "300007", "300008", "300009", "300010", 
                   "300011", "300012", "300013", "300014", "300015",
                     "300016", "300017"]

ModBus1_UnitDivision = [10000,1000,100,100,10000,
                        1000,1000,1000,10,10,
                        1000,1000,1000,1000,1000,
                        1000,10000,1000,10000,1000,
                        1000,1000]

ModBus15_UnitDivision = [100,100,100,100,100,
                        100,100,100,100,100,
                        100,100,100,100,100,
                        1,1]


In [3]:
def access_dir(data_sub_dir, data_dir):
    """
    Function is to generate dataset directory path based on the user defined process setting
    This is the primary acquired data directory (raw) 
    :param data_sub_dir # Choose the sub folder to get access to the desired dataset
    :param data_dir: # Choose the end directory containing desired data  
    :return: directory to access data
    """
    # get current directory
    currentpath = os.getcwd()
    # get parent data directory
    data_par_dir = os.path.abspath(os.path.join(currentpath, os.pardir))
    
    # Get the path to image dataset based on choices    
    d_dir = os.path.join(data_par_dir, (data_sub_dir + "/" + data_dir))
    
    return d_dir

def read_json_file(file):
    with open(file, "r") as r:
        response = r.read()
        response = response.replace('\n', '')
        response = response.replace('}{', '},{')
        response = "[" + response + "]"
        return json.loads(response)


def get_sensor_measurements(JSONLog,ADCSelect,ModBus1Select,ModBus15Select):
    
    # For each measurment sample, desired parameters are extracted with its corresponding units
    # selection of parameter is possible with user inputs
    ModBus1_val = []
    ModBus15_val = []
    ADCvalue = []
    
    # additional info
    ModBus1_unitsSelect =[]
    ModBus15_unitsSelect =[]
    ADC_unitSelect = []
    
    # get the time stamp for the measurements
    TimeStamp = JSONLog.get('timeStamp')[:-7]

    # get ADC sensor values
    if len(ADCSelect) == 1:
        ADCvalue = JSONLog.get('sensors').get('adc')[ADCSelect[0]]
        #ADC_unitSelect = ['DegC']
    else:
        for ADCIter in ADCSelect:
            ADCvalue.append(JSONLog.get('sensors').get('adc')[ADCIter])
            #ADC_unitSelect.append('DegC')
    
    # get ModBus sensor values - channel 1
    if len(ModBus1Select) == 1:
        ModBus1_tempval = JSONLog.get('sensors').get('modbus').get('1').get(ModBus1Select[0])        
        ModBus1_val.append(ModBus1_tempval/ModBus1_UnitDivision[ModBus1_address.index(ModBus1Select[0])])
        #ModBus1_unitsSelect.append(ModBus1_units[ModBus1_address.index(ModBus1Select[0])])
    
    else:
        for busIter in ModBus1Select:            
            ModBus1_tempval = JSONLog.get('sensors').get('modbus').get('1').get(busIter)
            ModBus1_val.append(ModBus1_tempval/ModBus1_UnitDivision[ModBus1_address.index(busIter)])  
            #ModBus1_unitsSelect.append(ModBus1_units[ModBus1_address.index(busIter)])
    
    # get ModBus sensor values - channel 15
    if len(ModBus15Select) == 1:
        ModBus15_tempval = JSONLog.get('sensors').get('modbus').get('15').get(ModBus15Select[0])        
        ModBus15_val.append(ModBus15_tempval/ModBus15_UnitDivision[ModBus1_address.index(ModBus15Select[0])])
        #ModBus15_unitsSelect.append(ModBus15_units[ModBus1_address.index(ModBus15Select[0])])
    else:
        for busIter in ModBus15Select:            
            ModBus15_tempval = JSONLog.get('sensors').get('modbus').get('15').get(busIter)
            ModBus15_val.append(ModBus15_tempval/ModBus15_UnitDivision[ModBus15_address.index(busIter)])
            #ModBus15_unitsSelect.append(ModBus15_units[ModBus15_address.index(busIter)]) 
    
    return ADCvalue,ModBus1_val,ModBus15_val,TimeStamp


def extractFeatures_toCSV(dataPath,dataSample,ADCSelect,ModBus1Select,ModBus15Select):

    # Extract individual features and save them as csv
    # Load the first json file for extracting available signals/feature list
    JsonF = os.path.join((dataPath+ "/") + dataSample)
    JsonF = read_json_file(file=JsonF)   

    # get the choosen parameter details
    #ModBus1_unitsSelected = [ModBus1_units[ModBus1_address.index(i)] for i in ModBus1Select]
    #ModBus15_unitsSelected = [ModBus15_units[ModBus15_address.index(i)] for i in ModBus15Select]
    ModBus1_infoSelected = [ModBus1_info[ModBus1_address.index(i)] for i in ModBus1Select]
    ModBus15_infoSelected = [ModBus15_info[ModBus15_address.index(i)] for i in ModBus15Select]
    ADC_infoSelected = [ADC_info[i] for i in ADCSelect]

    timeD = []
    ADCval = []
    Mod1val = []
    Mod15val = []

    # gather sensor measurements
    for i in range(len(JsonF)):
        JSONLog = JsonF[i]      
        # check if all the registers are logged for each measurments.
        # otherwise skip them for adding to the log
        # some measurements does not include all the registers for some reason
        if all(key in JSONLog['sensors']['modbus']['1'].keys() for key in ModBus1Select):
            if all(key in JSONLog['sensors']['modbus']['15'].keys() for key in ModBus15Select):
                A,B,C,D = get_sensor_measurements(JSONLog,ADCSelect,ModBus1Select,ModBus15Select)
        else:
            continue
        
        # append each measurement in a list
        timeD.append(D)
        ADCval.append(np.round(A,2))
        Mod1val.append(B)
        Mod15val.append(C)

    # create a pandas dataframe for all the measurements
    df = pd.DataFrame(ADCval, columns = [ADC_infoSelected]) 
    df[ModBus1_infoSelected] = Mod1val
    df[ModBus15_infoSelected] = Mod15val
    df.insert (0,'TimeStamp',timeD)
    
    # returns JSON object as a dictionary      
    return df

def get_state(to_process_data, df):
    # add colomn for machine state
    if 'Normal' in to_process_data or 'normal' in to_process_data:
        df['machine_state'] = 'normal'
    elif 'misalign' in to_process_data:
        df['machine_state'] = 'misalign'
    else:
        df['machine_state'] = 'unknown'

In [4]:
def process_raw_data(raw_data_path, to_process_data):
    print(f"Currently working on file: {to_process_data}")
    # get the readings from the data
    df = extractFeatures_toCSV(raw_data_path, to_process_data, ADCSelect, ModBus1Select, ModBus15Select)
    # add column for machine state
    get_state(to_process_data, df)
    
    return df

def process_folder(folder_name, result_dir, data_sub_dir='code/data/'):
    print(f"Currently working on folder: {folder_name}")

    data_sub_dir = data_sub_dir + folder_name
    
    # create folders for the results
    os.mkdir(result_dir + 'processed/' + folder_name)
    os.mkdir(result_dir + 'graph/' + folder_name)
    os.mkdir(result_dir + 'histogram/' + folder_name)
    
    # get all the raw data in the folder except the hidden files
    raw_data_list = [f for f in os.listdir(data_sub_dir + "/raw/") if not f.startswith('.')]
    for raw_data in raw_data_list:
        raw_data_path = access_dir(f'code/{data_sub_dir}', data_dir='raw')
        
        df = process_raw_data(raw_data_path, raw_data)
        
        # save the processed data
        save_name_csv = result_dir + 'processed/' + folder_name + '/' + raw_data[:-4] + '_res.csv'
        save_name_image = result_dir + 'graph/' + folder_name + '/' + raw_data[:-4] + '.png'
        save_name_histogram = result_dir + 'histogram/' + folder_name + '/' + raw_data[:-4] + '.png'
        
        df.to_csv(save_name_csv, index=False)
        
        df = pd.read_csv(save_name_csv, parse_dates=True)

        df['Time(sec)'] = df.index*6
        df[100:].plot.line(x='Time(sec)',figsize=(20, 25), subplots=True)
        plt.savefig(save_name_image)
        
        df.hist(layout=(8, 5), figsize=(20, 20), sharey=False, sharex=False, bins=50,grid=False)
        plt.savefig(save_name_histogram)
        
        plt.close('all')
        

In [5]:
def process_data(d):        
    # folder to save
    result = f'analysis/{d}_data/'
    data_sub = f'data/{d}/' 
    # reset results_data folder
    os.system('rm -rf '+ result)
    os.mkdir(result)
    os.mkdir(result + 'processed')
    os.mkdir(result + 'graph')
    os.mkdir(result + 'histogram')
    
    results_dir = os.getcwd() + '/' + result
    folder_to_use = [f for f in os.listdir(data_sub) if not f.startswith('.')]
            
    # process the folders
    for folder in folder_to_use:
        process_folder(folder, results_dir, data_sub)
    

In [6]:
ModBus1Select = ModBus1_address  #["405202","405204", "405206","405209","405210","405211","405212","405221","405222"] # ModBus1_address 
ModBus15Select = ModBus15_address # ["300001", "300004","300007","300010","300013"] #ModBus15_address
ADCSelect = []
if not os.path.exists('analysis'):
    os.mkdir('analysis')

# folder with the data
data_dir = os.listdir('data/')
# data_dir = []
for d in data_dir:
    print(f"Currently working on data: {d}")
    if not d.startswith('.'):
        process_data(d)

Currently working on data: ready
Currently working on folder: 110124
Currently working on file: 12-data_Normal_25Hz.txt
Currently working on file: 13-data_Normal_25Hz.txt
Currently working on file: 15-data_Normal_25Hz.txt
Currently working on file: 10-data_Normal_25Hz.txt
Currently working on file: 09-data_Normal_25Hz.txt
Currently working on file: 14-data_Normal_25Hz.txt
Currently working on file: 11-data_Normal_25Hz.txt
Currently working on folder: 120124
Currently working on file: 10-data_Normal_15Hz.txt
Currently working on file: 11-data_Normal_15Hz.txt
Currently working on file: 14-data_Normal_15Hz.txt
Currently working on file: 09-data_Normal_15Hz.txt
Currently working on file: 12-data_Normal_15Hz.txt
Currently working on file: 13-data_Normal_15Hz.txt
Currently working on folder: 210324
Currently working on file: 09-data_misalign_25Hz.txt
Currently working on file: 12-data_misalign_25Hz.txt
Currently working on file: 11-data_misalign_25Hz.txt
Currently working on file: 10-data_mi

# 2. Dataset Preparation